# Comparação entre Fonte_A (PDF) e Fonte_B (Numbers)

Este notebook localiza os arquivos dentro de `Dados/`, lê a **Fonte_A** em PDF e a **Fonte_B** em Numbers, normaliza os dados e executa as verificações solicitadas. Ao final, ele gera o arquivo `comparacao.md` na mesma pasta dos arquivos de entrada.

As comparações foram separadas em células para permitir execução isolada dos blocos de **Fonte_A**, **Fonte_B** e **Fonte_A_B**. Neste caderno, não há regras exclusivas para a Fonte_B, então a célula correspondente apenas registra essa condição.

In [ ]:
import re
import subprocess
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
import pdfplumber
from IPython.display import Markdown, display
from numbers_parser import Document

REPORT_COLUMNS = [
    "codigo",
    "nome_turma",
    "valor_fonte_a",
    "valor_fonte_b",
    "tipo_inconsistencia",
]
DAY_COLUMNS = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sab"]
DAY_POSITIONS = {
    "Seg": 523.36,
    "Ter": 553.88,
    "Qua": 582.04,
    "Qui": 616.16,
    "Sex": 646.52,
    "Sab": 676.92,
}
SCHEDULE_RE = re.compile(r"^\d{1,2}/\d{1,2}C?$")
COURSE_RE = re.compile(r"^[A-Z]{3}(?:-[A-Z])?$")
PLACEHOLDERS = {
    "_não_",
    "_NAO_",
    "_NÃO_",
    "_PSPS",
    "_NÃO OFERTA_",
    "_OfertarSoNoite_",
    "nan",
    "NaN",
}


def clean_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def strip_accents(text):
    return "".join(
        char
        for char in unicodedata.normalize("NFKD", clean_text(text).lower())
        if not unicodedata.combining(char)
    )


def split_mixed_token(token):
    token = clean_text(token)
    match = re.match(r"^(\d{3,6})([A-Za-zÀ-ÿ].*)$", token)
    if match:
        return [match.group(1), match.group(2)]
    return [token] if token else []


def format_schedule_map(schedule_map):
    parts = []
    for day in DAY_COLUMNS:
        values = [clean_text(value) for value in schedule_map.get(day, []) if clean_text(value)]
        if values:
            parts.append(f"{day} {', '.join(values)}")
    return " | ".join(parts)


def normalize_course_key(value):
    parts = re.split(r"\s*[|,]\s*", clean_text(value))
    bases = sorted({part.split("-")[0].strip() for part in parts if part.strip()})
    return " | ".join(bases)


def empty_report():
    return pd.DataFrame(columns=REPORT_COLUMNS)


def report_row(codigo, nome_turma, valor_fonte_a, valor_fonte_b, tipo):
    return {
        "codigo": clean_text(codigo),
        "nome_turma": clean_text(nome_turma),
        "valor_fonte_a": clean_text(valor_fonte_a) or "—",
        "valor_fonte_b": clean_text(valor_fonte_b) or "—",
        "tipo_inconsistencia": tipo,
    }


def resolve_dados_dir(base_dir: Path) -> Path:
    candidate = base_dir / "Dados"
    if candidate.is_dir():
        return candidate
    if candidate.is_file():
        script = f'''set aliasPath to POSIX file "{candidate}" as alias
tell application "Finder"
    set targetItem to original item of aliasPath
    return POSIX path of (targetItem as alias)
end tell'''
        result = subprocess.run(
            ["osascript"],
            input=script,
            text=True,
            capture_output=True,
            check=True,
        )
        return Path(result.stdout.strip())
    raise FileNotFoundError("Não foi possível localizar Dados/.")


def find_source_files(dados_dir: Path) -> tuple[Path, Path]:
    pdf_files = sorted(dados_dir.glob("*.pdf"), key=lambda path: path.stat().st_mtime)
    numbers_files = sorted(dados_dir.glob("*.numbers"), key=lambda path: path.stat().st_mtime)
    if not pdf_files or not numbers_files:
        raise FileNotFoundError("A pasta Dados/ precisa conter ao menos um PDF e um arquivo Numbers.")
    return pdf_files[-1], numbers_files[-1]


def group_lines(words, tolerance: float = 1.0):
    lines = []
    for word in sorted(words, key=lambda item: (item["top"], item["x0"])):
        text = clean_text(word["text"])
        if not text:
            continue
        for line in lines:
            if abs(line["top"] - word["top"]) <= tolerance:
                line["words"].append({"text": text, "x0": word["x0"], "x1": word["x1"]})
                break
        else:
            lines.append({"top": word["top"], "words": [{"text": text, "x0": word["x0"], "x1": word["x1"]}]})
    for line in lines:
        line["words"].sort(key=lambda item: item["x0"])
    return sorted(lines, key=lambda item: item["top"])


def parse_pdf_main_line(words):
    if len(words) < 6:
        return None

    first_four = [word["text"] for word in words[:4]]
    if not (
        first_four[0].endswith(".")
        and first_four[1].endswith(".")
        and first_four[2].endswith(".")
        and re.fullmatch(r"\d{3}-\d", first_four[3])
    ):
        return None

    cleaned_words = []
    for word in words[4:]:
        for piece in split_mixed_token(word["text"]):
            cleaned_words.append({"text": piece, "x0": word["x0"], "x1": word["x1"]})

    tail_words = [word for word in cleaned_words if word["x0"] >= 705]
    credit_words = [word for word in cleaned_words if 485 <= word["x0"] < 520]
    if len(tail_words) < 4 or len(credit_words) < 2:
        return None

    name_tokens = []
    professor_tokens = []
    professor_code_seen = False
    for word in [item for item in cleaned_words if 120 <= item["x0"] < 470]:
        if not professor_code_seen and re.fullmatch(r"\d{3,6}", word["text"]):
            professor_code_seen = True
            continue
        if professor_code_seen:
            professor_tokens.append(word["text"])
        else:
            name_tokens.append(word["text"])

    schedule_map = defaultdict(list)
    for word in cleaned_words:
        if 515 <= word["x0"] < 705 and SCHEDULE_RE.fullmatch(word["text"]):
            day = min(DAY_POSITIONS, key=lambda key: abs(DAY_POSITIONS[key] - word["x0"]))
            schedule_map[day].append(word["text"])

    return {
        "codigo": clean_text(" ".join(first_four)),
        "nome": clean_text(" ".join(name_tokens)),
        "professor": clean_text(" ".join(professor_tokens)),
        "credito_teorico": int(float(credit_words[0]["text"])),
        "credito_pratico": int(float(credit_words[1]["text"])),
        "curso_entries": [
            {
                "curso": clean_text(tail_words[1]["text"]),
                "fase": clean_text(tail_words[2]["text"]),
                "grupo": clean_text(tail_words[3]["text"]),
            }
        ],
        "schedule_map": schedule_map,
    }


def parse_pdf_source(pdf_path: Path) -> pd.DataFrame:
    records = []
    current_record = None

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            lines = group_lines(page.extract_words(use_text_flow=False, keep_blank_chars=False))
            for line in lines:
                words = line["words"]
                line_text = clean_text(" ".join(word["text"] for word in words))
                if not line_text:
                    continue
                if any(
                    line_text.startswith(prefix)
                    for prefix in (
                        "DIVISÃO DE REGISTROS",
                        "Registros Acadêmicos",
                        "Turma por Depto",
                        "Departamento:",
                        "Turma Código",
                        "Núcleo de Informática",
                    )
                ) or "Créditos" in line_text:
                    continue

                parsed = parse_pdf_main_line(words)
                if parsed is not None:
                    current_record = parsed
                    records.append(current_record)
                    continue

                if current_record is None:
                    continue

                schedule_words = [
                    word for word in words if 515 <= word["x0"] < 705 and SCHEDULE_RE.fullmatch(word["text"])
                ]
                if schedule_words:
                    for word in schedule_words:
                        day = min(DAY_POSITIONS, key=lambda key: abs(DAY_POSITIONS[key] - word["x0"]))
                        current_record["schedule_map"][day].append(word["text"])
                    continue

                tail_words = [word for word in words if word["x0"] >= 726]
                if len(tail_words) >= 3 and COURSE_RE.fullmatch(tail_words[0]["text"]):
                    current_record["curso_entries"].append(
                        {
                            "curso": clean_text(tail_words[0]["text"]),
                            "fase": clean_text(tail_words[1]["text"]),
                            "grupo": clean_text(tail_words[2]["text"]),
                        }
                    )
                    continue

                left_text = clean_text(" ".join(word["text"] for word in words if word["x0"] < 300))
                if left_text:
                    current_record["nome"] = clean_text(f"{current_record['nome']} {left_text}")

    rows = []
    for record in records:
        row = {
            "codigo": record["codigo"],
            "nome": record["nome"],
            "professor": record["professor"],
            "credito_teorico": record["credito_teorico"],
            "credito_pratico": record["credito_pratico"],
            "curso": " | ".join(dict.fromkeys(item["curso"] for item in record["curso_entries"] if item["curso"])),
        }
        for day in DAY_COLUMNS:
            row[day] = " | ".join(record["schedule_map"].get(day, []))
        row["horario"] = format_schedule_map(record["schedule_map"])
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["codigo"]).reset_index(drop=True)


def sanitize_numbers_value(value):
    text = clean_text(value)
    if text in PLACEHOLDERS:
        return ""
    return text


def parse_numbers_source(numbers_path: Path) -> pd.DataFrame:
    document = Document(str(numbers_path))
    table = document.sheets[0].tables[0]
    rows = list(table.rows(values_only=True))
    dataframe = pd.DataFrame(rows[1:], columns=rows[0])
    dataframe = dataframe[dataframe["Código"].notna()].copy()

    for column in dataframe.columns:
        dataframe[column] = dataframe[column].map(sanitize_numbers_value)

    dataframe = dataframe.rename(
        columns={
            "Disciplina": "nome",
            "Código": "codigo",
            "Curso": "curso",
            "Fase": "fase",
            "Grupo": "grupo",
            "Cre.": "credito_total",
            "Professor": "professor",
        }
    )

    for day in ("Seg", "Ter", "Qua", "Qui", "Sex"):
        dataframe[day] = dataframe[day].map(sanitize_numbers_value).str.replace("-", "/", regex=False)

    dataframe["horario"] = dataframe.apply(
        lambda row: " | ".join(
            f"{day} {row[day]}" for day in ("Seg", "Ter", "Qua", "Qui", "Sex") if clean_text(row[day])
        ),
        axis=1,
    )

    return dataframe[
        ["codigo", "nome", "professor", "curso", "fase", "grupo", "credito_total", "Seg", "Ter", "Qua", "Qui", "Sex", "horario"]
    ].sort_values(["codigo"]).reset_index(drop=True)


def display_report(title, dataframe):
    display(Markdown(f"## {title}"))
    if dataframe.empty:
        display(Markdown("_Nenhuma inconsistência encontrada._"))
    else:
        display(Markdown(dataframe.to_markdown(index=False)))


def compare_fonte_a(fonte_a: pd.DataFrame, fonte_b: pd.DataFrame) -> pd.DataFrame:
    lookup_b = fonte_b.set_index("codigo", drop=False)
    rows = []

    sem_professor = fonte_a[fonte_a["professor"].map(clean_text).eq("")]
    for _, turma in sem_professor.iterrows():
        valor_b = lookup_b.at[turma["codigo"], "professor"] if turma["codigo"] in lookup_b.index else ""
        rows.append(report_row(turma["codigo"], turma["nome"], "Professor ausente", valor_b, "Fonte_A: turma sem professor"))

    concentrado = fonte_a[fonte_a["horario"].str.contains("C", na=False)]
    for _, turma in concentrado.iterrows():
        valor_b = lookup_b.at[turma["codigo"], "horario"] if turma["codigo"] in lookup_b.index else ""
        rows.append(report_row(turma["codigo"], turma["nome"], turma["horario"], valor_b, "Fonte_A: turma em concentrado"))

    credito_impar = fonte_a[(fonte_a["credito_teorico"] + fonte_a["credito_pratico"]) % 2 == 1]
    for _, turma in credito_impar.iterrows():
        valor_a = (
            f"Teo={turma['credito_teorico']}, Prat={turma['credito_pratico']}, "
            f"Soma={turma['credito_teorico'] + turma['credito_pratico']}"
        )
        valor_b = ""
        if turma["codigo"] in lookup_b.index:
            valor_b = f"Cre. total={lookup_b.at[turma['codigo'], 'credito_total']}"
        rows.append(report_row(turma["codigo"], turma["nome"], valor_a, valor_b, "Fonte_A: soma de créditos ímpar"))

    return pd.DataFrame(rows, columns=REPORT_COLUMNS)


def compare_fonte_a_b(fonte_a: pd.DataFrame, fonte_b: pd.DataFrame) -> pd.DataFrame:
    merged = fonte_a.merge(fonte_b, on="codigo", how="outer", suffixes=("_a", "_b"), indicator=True)
    rows = []

    for _, turma in merged.iterrows():
        codigo = turma["codigo"]
        nome_a = clean_text(turma.get("nome_a", ""))
        nome_b = clean_text(turma.get("nome_b", ""))
        nome_ref = nome_a or nome_b

        if turma["_merge"] == "left_only":
            rows.append(report_row(codigo, nome_ref, nome_a, "", "Fonte_A_B: turma ausente na Fonte_B"))
            continue
        if turma["_merge"] == "right_only":
            rows.append(report_row(codigo, nome_ref, "", nome_b, "Fonte_A_B: turma ausente na Fonte_A"))
            continue

        if strip_accents(nome_a) != strip_accents(nome_b):
            rows.append(report_row(codigo, nome_ref, nome_a, nome_b, "Fonte_A_B: nome divergente"))

        professor_a = clean_text(turma.get("professor_a", ""))
        professor_b = clean_text(turma.get("professor_b", ""))
        if strip_accents(professor_a) != strip_accents(professor_b):
            rows.append(report_row(codigo, nome_ref, professor_a, professor_b, "Fonte_A_B: professor divergente"))

        horario_a = clean_text(turma.get("horario_a", ""))
        horario_b = clean_text(turma.get("horario_b", ""))
        if horario_a != horario_b:
            rows.append(report_row(codigo, nome_ref, horario_a, horario_b, "Fonte_A_B: horário divergente"))

        curso_a = clean_text(turma.get("curso_a", ""))
        curso_b = clean_text(turma.get("curso_b", ""))
        if normalize_course_key(curso_a) != normalize_course_key(curso_b):
            rows.append(report_row(codigo, nome_ref, curso_a, curso_b, "Fonte_A_B: curso divergente"))

    return pd.DataFrame(rows, columns=REPORT_COLUMNS)


def build_markdown_report(relatorio: pd.DataFrame, pdf_path: Path, numbers_path: Path) -> str:
    lines = [
        "# Relatório de comparação",
        "",
        f"- **Fonte_A (PDF):** `{pdf_path.name}`",
        f"- **Fonte_B (Numbers):** `{numbers_path.name}`",
        f"- **Total de inconsistências:** {len(relatorio)}",
        "",
    ]
    if relatorio.empty:
        lines.append("Nenhuma inconsistência encontrada.")
        return "\n".join(lines)

    resumo = relatorio.groupby("tipo_inconsistencia").size().reset_index(name="quantidade")
    lines.extend([
        "## Quantitativo",
        "",
        resumo.to_markdown(index=False),
        "",
        "## Detalhes",
        "",
        relatorio.to_markdown(index=False),
    ])
    return "\n".join(lines)

In [ ]:
PROJECT_DIR = Path.cwd()
DADOS_DIR = resolve_dados_dir(PROJECT_DIR)
PDF_PATH, NUMBERS_PATH = find_source_files(DADOS_DIR)

fonte_a = parse_pdf_source(PDF_PATH)
fonte_b = parse_numbers_source(NUMBERS_PATH)

info = f"""
**Pasta de dados:** `{DADOS_DIR}`  
**Fonte_A:** `{PDF_PATH.name}` ({len(fonte_a)} turmas)  
**Fonte_B:** `{NUMBERS_PATH.name}` ({len(fonte_b)} turmas)
"""

display(Markdown(info))
display(Markdown("### Amostra da Fonte_A"))
display(fonte_a.head())
display(Markdown("### Amostra da Fonte_B"))
display(fonte_b.head())

In [ ]:
relatorio_fonte_a = compare_fonte_a(fonte_a, fonte_b)
display_report("Comparações exclusivas da Fonte_A", relatorio_fonte_a)

In [ ]:
relatorio_fonte_b = empty_report()
display(Markdown("## Comparações exclusivas da Fonte_B"))
display(Markdown("_Nenhuma regra exclusiva da Fonte_B foi solicitada neste caderno._"))

In [ ]:
relatorio_fonte_a_b = compare_fonte_a_b(fonte_a, fonte_b)
display_report("Comparações entre Fonte_A e Fonte_B", relatorio_fonte_a_b)

In [ ]:
relatorio_final = pd.concat(
    [relatorio_fonte_a, relatorio_fonte_b, relatorio_fonte_a_b],
    ignore_index=True,
).sort_values(["tipo_inconsistencia", "codigo"]).reset_index(drop=True)

comparacao_md = build_markdown_report(relatorio_final, PDF_PATH, NUMBERS_PATH)
REPORT_PATH = DADOS_DIR / "comparacao.md"
REPORT_PATH.write_text(comparacao_md, encoding="utf-8")

display(Markdown(comparacao_md))
print(f"Relatório salvo em: {REPORT_PATH}")